# Mimicry-pair recovery

Does visual similarity place published mimicry partners near each other? As on the
species pages, a species' dorsal UNICOM centroid queries every image, and each species
scores its nearest image, as `similarity run` does. Ventral centroids are not queried.

Every species in the collection is ranked, so a partner outside the site's top ten still
has a measured rank. Pairs come from `analyses/data/mimicry_pairs.csv`, with their
publications and DOIs. Run with the backend stopped.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "backend/app/configs/config.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
from analyses.helpers.mimicry import describe, mimicry_results, recovery_figure
from analyses.helpers.publication import export_figure, load_settings, publication_style

settings = load_settings(ROOT)
publication_style()

N_PERMUTATIONS = 10_000
SEED = 0

## Rank and test every pair

One dorsal centroid per query species; every collection species is scored by its nearest
UNICOM image on either side. Each species' representative image is its dorsal image nearest
its own centroid.

In [ ]:
results = mimicry_results(settings, ROOT, N_PERMUTATIONS, SEED)
print(describe(results))

## Tested pairs and species

Published pairs and species against those tested. Tested species are counted by accepted
name, so a published name the collection files under another species counts once; every
tested species is ranked against all species in the collection.

In [ ]:
results.counts()

## Pairs and coverage

Species are MATCHED accepted names. A published name that is not itself accepted resolves
through the images recorded under it, when they agree on one accepted species (for example
*Adelpha bredowii* → *Limenitis bredowii*); `resolution` shows how each member resolved.
A pair is tested only when both members resolve to distinct species with dorsal images. Pairs not in the
collection are left out of the result tables; the summary above names them and why.

In [ ]:
results.coverage

## Partner ranks

`site_listed` marks partners the site's stored top-ten list already shows; `recovery` marks
pairs that are mutual top ten.

In [ ]:
results.ranks.drop(columns=["mimicry_type"])

In [ ]:
results.recovery

## Permutation tests

The statistic is the mean partner percentile over both query directions of every tested
pair (1 = nearest, 0.5 = chance). Each permutation keeps the queries and draws a new
partner per row:

- **Random**: any other collection species.
- **Congeners**: for pairs within one genus, another congener of the query, since close
  relatives look alike without mimicry; pairs across genera keep the random pool.

Directions of one pair, and species shared by several pairs, are not independent; the
p-values describe these pairs against these nulls, not a population of mimicry pairs.

In [ ]:
results.permutations

## Combined figure

A) Partner rank from the dorsal centroids. Each pair shows both species'
representative dorsal images (the image nearest the species' own centroid), first species
farther from the axis. Filled markers query with the pair's first species, hollow ones with
its second; the dashed line marks the site's top ten and the dotted line the chance median.
Shaded, bold pairs are mutual top ten: each species is among the other's ten most
similar. B) Observed mean partner percentile against each null's mean and
95% interval. Pair labels end in the mimicry type: (M) Müllerian, (B) Batesian, (M*)
Müllerian or context-dependent.

In [ ]:
fig = recovery_figure(results, results.pictures(ROOT))
export_figure(fig, settings, "mimicry_recovery", results.exports())
plt.show()
plt.close(fig)